In [ ]:
import pymupdf
import unicodedata, re
import torch
from sentence_transformers import SentenceTransformer

In [ ]:
"""
The purpose of this notebook is to test text embedding and searching.
Text cleaning and chunking is purposefully naive to quickly demo mostly the speed
of inference and "compilation" of the program, i.e. assigning model and text embeddings.
I also tested few other pre-trained multilingual models, but in this context
intfloat/multilingual-e5-small faired better than others.

For a quick test the results are promising.
"""

In [ ]:
path = r"C:\Users\Heikki\Documents\cost_estimation_ml\projektit\002_rak_selostukset\090_RAKENNUSSELOSTUS.pdf"

def clean_text(text):
    """
    Clean basic job description document text.
    """
    text = unicodedata.normalize('NFKC', text)
    text = text.replace('\n', '').strip()
    text = re.sub(r'\s{2,}', ' ', text)
    text = re.sub(r'[^\w\d]{3,}', ' ', text)

    return text

# lets pick one longer ~80 page document and simple-parse the text input

doc = pymupdf.open(path)
text = [p.get_text() for p in doc]
text = "\n".join(text)
text = clean_text(text)

# do a naive chunking for the document text

chunk_size = int(len(text) / 250)
chunk_overlap = int(0.1 * chunk_size)

start = 0
chunks = {}
i = 0
while start < len(text):
    chunk = text[start:(start + chunk_size + chunk_overlap)]
    chunks[f'chunk_{i}'] = chunk
    start += chunk_size
    i += 1

chunk_vals = [i for i in chunks.values()]



In [72]:
model1 = SentenceTransformer("intfloat/multilingual-e5-small")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5901.75it/s]


In [73]:
chunk_embeddings = model1.encode(chunk_vals, convert_to_tensor=True, normalize_embeddings=True)

query = "mitä väliseinistä sanotaan?"
query_embedding = model1.encode(query, convert_to_tensor=True, normalize_embeddings=True)

scores = chunk_embeddings @ query_embedding

top_k = torch.topk(scores, k=4)

for score, idx in zip(top_k.values, top_k.indices):
    idx = idx.item()
    score = score.item()

    print(chunk_vals[idx])

istävän seinän tulee ulottua aina samanlaisena rajoittaviin rakenteisiin saakka, myös alakaton yläpuolella. Kaksikerroksisen levyrakenteisen seinän levyjen saumat on limitettävä. Ääntä eristävät seinät, ikkunat ja ovet tiivistetään toisiaan sekä ympäröiviä ja lävistäviä rakenteita vastaan täysin ilmatiiviiksi. Väliseinisä mm pistorasiat eivät saa olla kohdakkain. Mikäli ne joudutaan kuitenkin asentamaan kohdakkain tulee ääneneristävyyden varmistamiseksi käyttää kipsivalua. Rakenteiden tulee olla tiiviitä ja peitettyjä myös sellaisissa paikoissa, jotka eivät jää näkyviin kuten alakattojen yläpuolella, peittävien pintaverhousten, kalusteiden ja peitelistojen takana jne. Tiivistys tulee tehdä akustisilla tiivistyskittauksilla (AkustoSael Työn aikaiset aukot suljetaan ensisijaisesti seinämän rakennusmateriaalilla, kuitenkin liikkuvassa rakenteessa joustavalla, liikkeen sallivalla tiivistysmassalla. LVIS-laitteiden upotus ja läpivienti on tehtävä siten, ettei ääneneristävyys niiden kautta h